In [1]:
import requests
import pandas as pd
import time
import json
from pathlib import Path

# ── Clé API ───────────────────────────────────────────────
API_KEY  = "DEMO_KEY"  # ← remplace par ta vraie clé
BASE_URL = "https://api.nal.usda.gov/fdc/v1"

# ── Charger les 101 classes ───────────────────────────────
with open("C:/SmartMeal/src/config.json") as f:
    config = json.load(f)

CLASSES_101 = config["classes"]
print(f"✅ {len(CLASSES_101)} classes à enrichir")
print(f"Exemples : {CLASSES_101[:5]}")

✅ 101 classes à enrichir
Exemples : ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare']


In [2]:
def search_usda(query, api_key=API_KEY):
    """Recherche un aliment dans la base USDA."""
    try:
        url    = f"{BASE_URL}/foods/search"
        params = {
            "query"    : query,
            "api_key"  : api_key,
            "pageSize" : 1,
            "dataType" : "SR Legacy,Foundation"
        }
        r = requests.get(url, params=params, timeout=10)
        if r.status_code == 200:
            data = r.json()
            if data["foods"]:
                food = data["foods"][0]
                nutrients = {}
                nutrient_map = {
                    "Energy"                        : "calories",
                    "Protein"                       : "protein_g",
                    "Total lipid (fat)"             : "fat_g",
                    "Carbohydrate, by difference"   : "carbs_g",
                    "Fiber, total dietary"          : "fiber_g",
                    "Sugars, total"                 : "sugar_g",
                    "Sodium, Na"                    : "sodium_mg",
                }
                for n in food.get("foodNutrients", []):
                    name = n.get("nutrientName","")
                    if name in nutrient_map:
                        nutrients[nutrient_map[name]] = round(n.get("value",0), 2)
                return nutrients, food["description"]
        elif r.status_code == 429:
            print(f"   ⏳ Rate limit — pause 60s...")
            time.sleep(60)
            return search_usda(query, api_key)
    except Exception as e:
        print(f"   ❌ Erreur : {e}")
    return {}, "Non trouvé"

# Test
nuts, desc = search_usda("pizza cheese")
print(f"Test pizza : {desc}")
print(f"Nutriments : {nuts}")

   ⏳ Rate limit — pause 60s...
   ⏳ Rate limit — pause 60s...
   ⏳ Rate limit — pause 60s...
   ⏳ Rate limit — pause 60s...
   ⏳ Rate limit — pause 60s...
   ⏳ Rate limit — pause 60s...
   ⏳ Rate limit — pause 60s...
   ⏳ Rate limit — pause 60s...
   ⏳ Rate limit — pause 60s...
   ⏳ Rate limit — pause 60s...
   ⏳ Rate limit — pause 60s...


KeyboardInterrupt: 

In [3]:
# ── Base nutritionnelle complète 101 classes ──────────────
# Valeurs pour 100g — sources USDA + littérature
NUTRITION_DB = {
    "apple_pie"           : {"calories":237,"protein_g":2.0, "fat_g":11.0,"carbs_g":34.0,"fiber_g":1.5,"sugar_g":16.0,"sodium_mg":169},
    "baby_back_ribs"      : {"calories":297,"protein_g":22.0,"fat_g":23.0,"carbs_g":0.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":280},
    "baklava"             : {"calories":428,"protein_g":5.6, "fat_g":25.0,"carbs_g":46.0,"fiber_g":2.0,"sugar_g":28.0,"sodium_mg":230},
    "beef_carpaccio"      : {"calories":150,"protein_g":20.0,"fat_g":8.0, "carbs_g":0.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":180},
    "beef_tartare"        : {"calories":196,"protein_g":20.0,"fat_g":12.0,"carbs_g":2.0, "fiber_g":0.0,"sugar_g":1.0, "sodium_mg":420},
    "beet_salad"          : {"calories":74, "protein_g":2.5, "fat_g":3.0, "carbs_g":10.0,"fiber_g":2.5,"sugar_g":7.0, "sodium_mg":180},
    "beignets"            : {"calories":390,"protein_g":5.0, "fat_g":20.0,"carbs_g":48.0,"fiber_g":1.0,"sugar_g":15.0,"sodium_mg":280},
    "bibimbap"            : {"calories":490,"protein_g":26.0,"fat_g":12.0,"carbs_g":69.0,"fiber_g":4.0,"sugar_g":6.0, "sodium_mg":890},
    "bread_pudding"       : {"calories":197,"protein_g":5.8, "fat_g":7.5, "carbs_g":28.0,"fiber_g":0.7,"sugar_g":14.0,"sodium_mg":224},
    "breakfast_burrito"   : {"calories":220,"protein_g":10.0,"fat_g":10.0,"carbs_g":24.0,"fiber_g":2.0,"sugar_g":2.0, "sodium_mg":480},
    "bruschetta"          : {"calories":195,"protein_g":5.5, "fat_g":7.5, "carbs_g":27.0,"fiber_g":2.5,"sugar_g":3.0, "sodium_mg":390},
    "caesar_salad"        : {"calories":190,"protein_g":8.0, "fat_g":15.0,"carbs_g":8.0, "fiber_g":2.0,"sugar_g":2.0, "sodium_mg":420},
    "cannoli"             : {"calories":326,"protein_g":7.0, "fat_g":19.0,"carbs_g":32.0,"fiber_g":0.5,"sugar_g":18.0,"sodium_mg":150},
    "caprese_salad"       : {"calories":166,"protein_g":10.0,"fat_g":12.0,"carbs_g":4.0, "fiber_g":0.5,"sugar_g":3.0, "sodium_mg":430},
    "carrot_cake"         : {"calories":415,"protein_g":4.5, "fat_g":21.0,"carbs_g":54.0,"fiber_g":1.5,"sugar_g":36.0,"sodium_mg":310},
    "ceviche"             : {"calories":100,"protein_g":15.0,"fat_g":2.5, "carbs_g":5.0, "fiber_g":1.0,"sugar_g":2.0, "sodium_mg":520},
    "cheesecake"          : {"calories":321,"protein_g":5.5, "fat_g":23.0,"carbs_g":25.0,"fiber_g":0.3,"sugar_g":18.0,"sodium_mg":290},
    "cheese_plate"        : {"calories":350,"protein_g":22.0,"fat_g":28.0,"carbs_g":2.0, "fiber_g":0.0,"sugar_g":0.5, "sodium_mg":620},
    "chicken_curry"       : {"calories":238,"protein_g":18.0,"fat_g":14.0,"carbs_g":10.0,"fiber_g":2.0,"sugar_g":4.0, "sodium_mg":580},
    "chicken_quesadilla"  : {"calories":278,"protein_g":16.0,"fat_g":13.0,"carbs_g":24.0,"fiber_g":1.5,"sugar_g":2.0, "sodium_mg":520},
    "chicken_wings"       : {"calories":290,"protein_g":27.0,"fat_g":19.0,"carbs_g":0.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":370},
    "chocolate_cake"      : {"calories":371,"protein_g":5.0, "fat_g":16.0,"carbs_g":54.0,"fiber_g":2.5,"sugar_g":36.0,"sodium_mg":290},
    "chocolate_mousse"    : {"calories":260,"protein_g":5.0, "fat_g":18.0,"carbs_g":22.0,"fiber_g":1.5,"sugar_g":18.0,"sodium_mg":55},
    "churros"             : {"calories":367,"protein_g":5.5, "fat_g":17.0,"carbs_g":49.0,"fiber_g":2.0,"sugar_g":14.0,"sodium_mg":390},
    "clam_chowder"        : {"calories":112,"protein_g":5.0, "fat_g":4.5, "carbs_g":13.0,"fiber_g":0.8,"sugar_g":2.5, "sodium_mg":590},
    "club_sandwich"       : {"calories":285,"protein_g":19.0,"fat_g":13.0,"carbs_g":24.0,"fiber_g":2.0,"sugar_g":3.5, "sodium_mg":680},
    "crab_cakes"          : {"calories":178,"protein_g":15.0,"fat_g":9.5, "carbs_g":8.0, "fiber_g":0.5,"sugar_g":1.0, "sodium_mg":490},
    "creme_brulee"        : {"calories":275,"protein_g":4.5, "fat_g":18.0,"carbs_g":24.0,"fiber_g":0.0,"sugar_g":22.0,"sodium_mg":65},
    "croque_madame"       : {"calories":310,"protein_g":18.0,"fat_g":18.0,"carbs_g":20.0,"fiber_g":1.0,"sugar_g":3.0, "sodium_mg":720},
    "cup_cakes"           : {"calories":368,"protein_g":4.5, "fat_g":16.0,"carbs_g":53.0,"fiber_g":0.8,"sugar_g":35.0,"sodium_mg":290},
    "deviled_eggs"        : {"calories":149,"protein_g":9.0, "fat_g":11.0,"carbs_g":2.0, "fiber_g":0.0,"sugar_g":1.0, "sodium_mg":330},
    "donuts"              : {"calories":452,"protein_g":5.0, "fat_g":25.0,"carbs_g":52.0,"fiber_g":1.5,"sugar_g":22.0,"sodium_mg":370},
    "dumplings"           : {"calories":185,"protein_g":8.5, "fat_g":6.5, "carbs_g":24.0,"fiber_g":1.5,"sugar_g":2.0, "sodium_mg":450},
    "edamame"             : {"calories":121,"protein_g":11.0,"fat_g":5.2, "carbs_g":8.9, "fiber_g":5.2,"sugar_g":2.2, "sodium_mg":6},
    "eggs_benedict"       : {"calories":289,"protein_g":16.0,"fat_g":19.0,"carbs_g":14.0,"fiber_g":0.7,"sugar_g":2.0, "sodium_mg":680},
    "escargots"           : {"calories":130,"protein_g":19.0,"fat_g":5.0, "carbs_g":2.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":290},
    "falafel"             : {"calories":333,"protein_g":13.0,"fat_g":18.0,"carbs_g":32.0,"fiber_g":7.0,"sugar_g":4.0, "sodium_mg":585},
    "filet_mignon"        : {"calories":227,"protein_g":28.0,"fat_g":13.0,"carbs_g":0.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":59},
    "fish_and_chips"      : {"calories":310,"protein_g":16.0,"fat_g":15.0,"carbs_g":29.0,"fiber_g":2.5,"sugar_g":1.0, "sodium_mg":480},
    "foie_gras"           : {"calories":462,"protein_g":11.0,"fat_g":44.0,"carbs_g":5.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":690},
    "french_fries"        : {"calories":312,"protein_g":3.4, "fat_g":15.0,"carbs_g":41.0,"fiber_g":3.8,"sugar_g":0.3, "sodium_mg":210},
    "french_onion_soup"   : {"calories":73, "protein_g":3.5, "fat_g":2.5, "carbs_g":9.0, "fiber_g":1.0,"sugar_g":4.0, "sodium_mg":640},
    "french_toast"        : {"calories":229,"protein_g":8.5, "fat_g":9.5, "carbs_g":27.0,"fiber_g":1.0,"sugar_g":8.0, "sodium_mg":330},
    "fried_calamari"      : {"calories":175,"protein_g":14.0,"fat_g":7.5, "carbs_g":12.0,"fiber_g":0.5,"sugar_g":0.5, "sodium_mg":290},
    "fried_rice"          : {"calories":163,"protein_g":4.5, "fat_g":4.5, "carbs_g":26.0,"fiber_g":1.0,"sugar_g":1.5, "sodium_mg":480},
    "frozen_yogurt"       : {"calories":127,"protein_g":3.5, "fat_g":2.0, "carbs_g":24.0,"fiber_g":0.0,"sugar_g":20.0,"sodium_mg":65},
    "garlic_bread"        : {"calories":350,"protein_g":8.0, "fat_g":16.0,"carbs_g":44.0,"fiber_g":2.0,"sugar_g":2.5, "sodium_mg":620},
    "gnocchi"             : {"calories":131,"protein_g":3.5, "fat_g":0.5, "carbs_g":28.0,"fiber_g":1.5,"sugar_g":0.5, "sodium_mg":330},
    "greek_salad"         : {"calories":110,"protein_g":3.5, "fat_g":8.0, "carbs_g":7.0, "fiber_g":2.0,"sugar_g":4.0, "sodium_mg":480},
    "grilled_cheese_sandwich": {"calories":350,"protein_g":15.0,"fat_g":20.0,"carbs_g":28.0,"fiber_g":1.5,"sugar_g":4.0,"sodium_mg":740},
    "grilled_salmon"      : {"calories":208,"protein_g":20.0,"fat_g":13.0,"carbs_g":0.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":59},
    "guacamole"           : {"calories":160,"protein_g":2.0, "fat_g":15.0,"carbs_g":9.0, "fiber_g":6.7,"sugar_g":0.7, "sodium_mg":360},
    "gyoza"               : {"calories":200,"protein_g":9.0, "fat_g":8.5, "carbs_g":23.0,"fiber_g":1.5,"sugar_g":2.0, "sodium_mg":470},
    "hamburger"           : {"calories":295,"protein_g":17.0,"fat_g":14.0,"carbs_g":24.0,"fiber_g":1.3,"sugar_g":6.0, "sodium_mg":396},
    "hot_and_sour_soup"   : {"calories":67, "protein_g":3.5, "fat_g":2.5, "carbs_g":8.0, "fiber_g":1.0,"sugar_g":2.0, "sodium_mg":730},
    "hot_dog"             : {"calories":290,"protein_g":11.0,"fat_g":17.0,"carbs_g":24.0,"fiber_g":1.0,"sugar_g":5.0, "sodium_mg":620},
    "huevos_rancheros"    : {"calories":210,"protein_g":11.0,"fat_g":12.0,"carbs_g":16.0,"fiber_g":3.0,"sugar_g":4.0, "sodium_mg":480},
    "hummus"              : {"calories":166,"protein_g":8.0, "fat_g":10.0,"carbs_g":14.0,"fiber_g":6.0,"sugar_g":0.5, "sodium_mg":379},
    "ice_cream"           : {"calories":207,"protein_g":3.5, "fat_g":11.0,"carbs_g":24.0,"fiber_g":0.0,"sugar_g":21.0,"sodium_mg":80},
    "lasagna"             : {"calories":166,"protein_g":10.0,"fat_g":7.5, "carbs_g":15.0,"fiber_g":1.5,"sugar_g":4.0, "sodium_mg":430},
    "lobster_bisque"      : {"calories":120,"protein_g":7.5, "fat_g":6.5, "carbs_g":9.0, "fiber_g":0.5,"sugar_g":3.0, "sodium_mg":560},
    "lobster_roll_sandwich": {"calories":290,"protein_g":18.0,"fat_g":12.0,"carbs_g":28.0,"fiber_g":1.5,"sugar_g":4.0,"sodium_mg":590},
    "macaroni_and_cheese" : {"calories":164,"protein_g":6.5, "fat_g":6.5, "carbs_g":20.0,"fiber_g":1.0,"sugar_g":2.5, "sodium_mg":370},
    "macarons"            : {"calories":404,"protein_g":7.0, "fat_g":15.0,"carbs_g":62.0,"fiber_g":1.0,"sugar_g":52.0,"sodium_mg":55},
    "miso_soup"           : {"calories":40, "protein_g":3.0, "fat_g":1.5, "carbs_g":4.5, "fiber_g":1.0,"sugar_g":1.0, "sodium_mg":630},
    "mussels"             : {"calories":172,"protein_g":24.0,"fat_g":4.5, "carbs_g":7.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":286},
    "nachos"              : {"calories":346,"protein_g":9.0, "fat_g":19.0,"carbs_g":36.0,"fiber_g":3.5,"sugar_g":2.0, "sodium_mg":560},
    "omelette"            : {"calories":154,"protein_g":10.0,"fat_g":12.0,"carbs_g":1.0, "fiber_g":0.0,"sugar_g":0.5, "sodium_mg":300},
    "onion_rings"         : {"calories":411,"protein_g":5.0, "fat_g":22.0,"carbs_g":48.0,"fiber_g":2.5,"sugar_g":5.0, "sodium_mg":430},
    "oysters"             : {"calories":81, "protein_g":9.5, "fat_g":2.5, "carbs_g":5.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":417},
    "pad_thai"            : {"calories":182,"protein_g":8.5, "fat_g":6.5, "carbs_g":24.0,"fiber_g":1.5,"sugar_g":5.0, "sodium_mg":580},
    "paella"              : {"calories":195,"protein_g":12.0,"fat_g":7.5, "carbs_g":21.0,"fiber_g":1.5,"sugar_g":2.0, "sodium_mg":490},
    "pancakes"            : {"calories":227,"protein_g":6.5, "fat_g":7.5, "carbs_g":35.0,"fiber_g":1.0,"sugar_g":8.0, "sodium_mg":400},
    "panna_cotta"         : {"calories":190,"protein_g":3.5, "fat_g":13.0,"carbs_g":16.0,"fiber_g":0.0,"sugar_g":14.0,"sodium_mg":45},
    "peking_duck"         : {"calories":337,"protein_g":19.0,"fat_g":28.0,"carbs_g":2.0, "fiber_g":0.0,"sugar_g":1.0, "sodium_mg":420},
    "pho"                 : {"calories":89, "protein_g":7.0, "fat_g":2.0, "carbs_g":11.0,"fiber_g":1.0,"sugar_g":2.5, "sodium_mg":470},
    "pizza"               : {"calories":266,"protein_g":11.0,"fat_g":10.0,"carbs_g":33.0,"fiber_g":2.3,"sugar_g":3.6, "sodium_mg":598},
    "pork_chop"           : {"calories":231,"protein_g":25.0,"fat_g":14.0,"carbs_g":0.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":62},
    "poutine"             : {"calories":280,"protein_g":9.0, "fat_g":14.0,"carbs_g":31.0,"fiber_g":2.5,"sugar_g":2.0, "sodium_mg":490},
    "prime_rib"           : {"calories":340,"protein_g":28.0,"fat_g":25.0,"carbs_g":0.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":68},
    "pulled_pork_sandwich": {"calories":280,"protein_g":22.0,"fat_g":11.0,"carbs_g":23.0,"fiber_g":1.5,"sugar_g":8.0, "sodium_mg":590},
    "ramen"               : {"calories":137,"protein_g":7.0, "fat_g":5.5, "carbs_g":16.0,"fiber_g":1.0,"sugar_g":2.0, "sodium_mg":640},
    "ravioli"             : {"calories":180,"protein_g":8.0, "fat_g":5.5, "carbs_g":25.0,"fiber_g":1.5,"sugar_g":2.5, "sodium_mg":380},
    "red_velvet_cake"     : {"calories":376,"protein_g":4.5, "fat_g":18.0,"carbs_g":51.0,"fiber_g":1.0,"sugar_g":34.0,"sodium_mg":310},
    "risotto"             : {"calories":166,"protein_g":4.5, "fat_g":5.5, "carbs_g":24.0,"fiber_g":0.5,"sugar_g":1.5, "sodium_mg":390},
    "samosa"              : {"calories":308,"protein_g":6.0, "fat_g":16.0,"carbs_g":36.0,"fiber_g":3.5,"sugar_g":3.0, "sodium_mg":490},
    "sashimi"             : {"calories":146,"protein_g":22.0,"fat_g":6.0, "carbs_g":0.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":55},
    "scallops"            : {"calories":111,"protein_g":20.0,"fat_g":1.5, "carbs_g":5.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":392},
    "seaweed_salad"       : {"calories":70, "protein_g":1.5, "fat_g":3.5, "carbs_g":8.0, "fiber_g":2.5,"sugar_g":2.0, "sodium_mg":530},
    "shrimp_and_grits"    : {"calories":198,"protein_g":14.0,"fat_g":8.0, "carbs_g":19.0,"fiber_g":1.0,"sugar_g":2.0, "sodium_mg":520},
    "spaghetti_bolognese" : {"calories":185,"protein_g":11.0,"fat_g":6.5, "carbs_g":22.0,"fiber_g":2.5,"sugar_g":5.0, "sodium_mg":380},
    "spaghetti_carbonara" : {"calories":339,"protein_g":14.0,"fat_g":16.0,"carbs_g":35.0,"fiber_g":2.0,"sugar_g":2.0, "sodium_mg":490},
    "spring_rolls"        : {"calories":138,"protein_g":4.5, "fat_g":5.5, "carbs_g":18.0,"fiber_g":1.5,"sugar_g":2.0, "sodium_mg":320},
    "steak"               : {"calories":271,"protein_g":26.0,"fat_g":18.0,"carbs_g":0.0, "fiber_g":0.0,"sugar_g":0.0, "sodium_mg":60},
    "strawberry_shortcake": {"calories":280,"protein_g":4.5, "fat_g":12.0,"carbs_g":39.0,"fiber_g":1.5,"sugar_g":22.0,"sodium_mg":185},
    "sushi"               : {"calories":143,"protein_g":5.8, "fat_g":3.2, "carbs_g":21.0,"fiber_g":0.6,"sugar_g":5.0, "sodium_mg":367},
    "tacos"               : {"calories":226,"protein_g":12.0,"fat_g":10.0,"carbs_g":22.0,"fiber_g":3.0,"sugar_g":3.0, "sodium_mg":440},
    "takoyaki"            : {"calories":200,"protein_g":9.0, "fat_g":9.5, "carbs_g":21.0,"fiber_g":1.0,"sugar_g":4.0, "sodium_mg":480},
    "tiramisu"            : {"calories":240,"protein_g":5.5, "fat_g":14.0,"carbs_g":23.0,"fiber_g":0.5,"sugar_g":16.0,"sodium_mg":120},
    "tuna_tartare"        : {"calories":134,"protein_g":22.0,"fat_g":4.5, "carbs_g":2.0, "fiber_g":0.5,"sugar_g":1.0, "sodium_mg":290},
    "waffles"             : {"calories":291,"protein_g":7.9, "fat_g":13.0,"carbs_g":37.0,"fiber_g":1.3,"sugar_g":6.0, "sodium_mg":400},
}

print(f"✅ Base nutritionnelle : {len(NUTRITION_DB)} aliments")
missing = [c for c in CLASSES_101 if c not in NUTRITION_DB]
print(f"⚠️  Manquants : {len(missing)} → {missing}")

✅ Base nutritionnelle : 101 aliments
⚠️  Manquants : 0 → []


In [4]:
def calculate_score(food_data):
    score   = 50
    cal     = food_data.get("calories",   0)
    protein = food_data.get("protein_g",  0)
    fat     = food_data.get("fat_g",      0)
    fiber   = food_data.get("fiber_g",    0)
    sugar   = food_data.get("sugar_g",    0)
    sodium  = food_data.get("sodium_mg",  0)

    if protein >= 20: score += 20
    elif protein >= 10: score += 10
    elif protein >= 5:  score += 5

    if fiber >= 5:  score += 15
    elif fiber >= 3: score += 10
    elif fiber >= 1: score += 5

    if 150 <= cal <= 300: score += 10
    if cal > 400:   score -= 20
    elif cal > 300: score -= 10

    if fat > 20:   score -= 15
    elif fat > 15: score -= 8

    if sugar > 15:  score -= 15
    elif sugar > 8: score -= 8

    if sodium > 600:  score -= 20
    elif sodium > 400: score -= 10
    elif sodium > 200: score -= 5

    return max(0, min(100, score))

def get_grade(score):
    if score >= 80: return "A","🟢","Excellent"
    if score >= 65: return "B","🟡","Bon"
    if score >= 50: return "C","🟠","Moyen"
    if score >= 35: return "D","🔴","Médiocre"
    return "E","⛔","Mauvais"

# ── Générer pour tous ─────────────────────────────────────
nutrition_rows = []
scores_rows    = []

for cls in CLASSES_101:
    if cls in NUTRITION_DB:
        n     = NUTRITION_DB[cls]
        score = calculate_score(n)
        g, e, l = get_grade(score)

        nutrition_rows.append({"class_name": cls, **n})
        scores_rows.append({
            "class_name": cls,
            "score"     : score,
            "grade"     : g,
            "emoji"     : e,
            "label"     : l
        })

df_nutrition_101 = pd.DataFrame(nutrition_rows)
df_scores_101    = pd.DataFrame(scores_rows).sort_values("score", ascending=False)

print(f"✅ {len(df_nutrition_101)} aliments avec nutrition")
print(f"✅ {len(df_scores_101)} aliments avec scores")
print(f"\n🏆 Top 10 :")
print(df_scores_101.head(10)[["class_name","score","grade"]].to_string(index=False))
print(f"\n⚠️  Bottom 5 :")
print(df_scores_101.tail(5)[["class_name","score","grade"]].to_string(index=False))

✅ 101 aliments avec nutrition
✅ 101 aliments avec scores

🏆 Top 10 :
          class_name  score grade
      beef_carpaccio     80     A
      grilled_salmon     80     A
           pork_chop     80     A
        filet_mignon     80     A
             edamame     75     B
             mussels     75     B
pulled_pork_sandwich     75     B
              hummus     75     B
               steak     72     B
           hamburger     70     B

⚠️  Bottom 5 :
 class_name  score grade
 cheesecake     10     E
    baklava      5     E
     donuts      5     E
  foie_gras      5     E
carrot_cake      0     E


In [5]:
# Sauvegarder les nouvelles bases
df_nutrition_101.to_csv("C:/SmartMeal/data/usda/nutrition_food101.csv", index=False)
df_scores_101.to_csv("C:/SmartMeal/data/usda/health_scores_101.csv",    index=False)

print("✅ nutrition_food101.csv sauvegardé")
print("✅ health_scores_101.csv sauvegardé")
print(f"\n📊 Stats :")
print(f"   Score moyen  : {df_scores_101['score'].mean():.1f}/100")
print(f"   Grade A      : {(df_scores_101['grade']=='A').sum()} aliments")
print(f"   Grade B      : {(df_scores_101['grade']=='B').sum()} aliments")
print(f"   Grade C      : {(df_scores_101['grade']=='C').sum()} aliments")
print(f"   Grade D/E    : {(df_scores_101['grade'].isin(['D','E'])).sum()} aliments")

✅ nutrition_food101.csv sauvegardé
✅ health_scores_101.csv sauvegardé

📊 Stats :
   Score moyen  : 50.2/100
   Grade A      : 4 aliments
   Grade B      : 31 aliments
   Grade C      : 25 aliments
   Grade D/E    : 41 aliments
